# Tomographic Reconstruction with JSON Metadata

This notebook demonstrates how to use the tomographic reconstruction code with the new JSON metadata format.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt

# Add the src directory to the path
sys.path.append('../src')

# Import our custom modules
from utils import calculate_optimal_volume_shape, calculate_memory_requirements

## Loading Data with JSON Metadata

First, let's load the projections and metadata from the JSON format.

In [ ]:
# Import the metadata handling module
from metadata_handling import load_data_with_json_metadata

# Set the path to your data directory
data_path = "../data/raw"

# Load data with JSON metadata (replace 'metadata.json' with your actual filename)
projections, angles, metadata, geometry = load_data_with_json_metadata(data_path, 'metadata.json')

## Examining the Metadata

Let's look at the important geometry information extracted from the metadata.

In [ ]:
# Extract key geometry information
if 'geometry' in metadata:
    geom = metadata['geometry']
    print("Geometry Information:")
    
    # Source-object distance (mm)
    if 'distanceSourceObject' in geom:
        print(f"Source-Object Distance: {geom['distanceSourceObject']} mm")
    
    # Object-detector distance (mm)
    if 'distanceObjectDetector' in geom:
        print(f"Object-Detector Distance: {geom['distanceObjectDetector']} mm")
    
    # Total distance (source to detector)
    if 'distanceSourceDetector' in geom:
        print(f"Source-Detector Distance: {geom['distanceSourceDetector']} mm")
    elif 'distanceSourceObject' in geom and 'distanceObjectDetector' in geom:
        total = geom['distanceSourceObject'] + geom['distanceObjectDetector']
        print(f"Source-Detector Distance: {total} mm")
    
    # Detector dimensions
    if 'detectorPixel' in geom:
        print(f"Detector Pixels: {geom['detectorPixel']}")
    
    # Object bounding box if available
    if 'objectBoundingBox' in geom:
        bbox = geom['objectBoundingBox']
        if 'sizeXYZ' in bbox:
            print(f"Object Size (XYZ): {bbox['sizeXYZ']} mm")
        if 'centerXYZ' in bbox:
            print(f"Object Center (XYZ): {bbox['centerXYZ']} mm")

# Check the projection angles
print(f"\nProjection Angles:")
print(f"Number of angles: {len(angles)}")
print(f"Angle range: {angles.min():.2f}° to {angles.max():.2f}°")

# Plot first few angles to check
plt.figure(figsize=(10, 5))
plt.plot(angles[:20], 'o-')
plt.xlabel('Projection Index')
plt.ylabel('Angle (degrees)')
plt.title('First 20 Projection Angles')
plt.grid(True)
plt.show()

## Examining the Projection Data

Let's look at the projection data dimensions and visualize some of the projections.

In [ ]:
# Print projection shape
print(f"Projection shape: {projections.shape}")

# Display a few projections
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
indices = [0, len(projections)//2, len(projections)-1]  # First, middle, last

for i, idx in enumerate(indices):
    axs[i].imshow(projections[idx], cmap='gray')
    axs[i].set_title(f"Projection at {angles[idx]:.1f}°")
    axs[i].axis('off')

plt.tight_layout()
plt.show()

## Calculate Optimal Volume Shape

Based on the projection data and metadata, let's calculate an optimal volume shape for reconstruction.

In [ ]:
# Calculate optimal volume shape based on projections and metadata
optimal_shape = calculate_optimal_volume_shape(projections, metadata)
print(f"Optimal volume shape: {optimal_shape}")

# Calculate memory requirements
memory_gb = calculate_memory_requirements(optimal_shape)
print(f"Estimated memory requirement: {memory_gb:.2f} GB")

# If memory is too large, suggest a downsampled shape
if memory_gb > 10:  # Arbitrary threshold of 10GB
    downsample_factor = 2  # Start with factor of 2
    
    # Find a downsample factor that brings memory below threshold
    while calculate_memory_requirements(tuple(dim//downsample_factor for dim in optimal_shape)) > 10 and downsample_factor < 8:
        downsample_factor += 1
    
    downsampled_shape = tuple(dim//downsample_factor for dim in optimal_shape)
    downsampled_memory = calculate_memory_requirements(downsampled_shape)
    
    print(f"\nWARNING: Memory requirement is high!")
    print(f"Suggested downsampled shape (factor {downsample_factor}): {downsampled_shape}")
    print(f"Downsampled memory requirement: {downsampled_memory:.2f} GB")

## Perform Reconstruction

Now let's perform a tomographic reconstruction using our updated code.

In [ ]:
# Import reconstruction module
from accelerated import filtered_backprojection, sirt_reconstruction, fdk_reconstruction

# Choose a reasonable volume_shape based on previous calculations
# If memory is limited, use the downsampled shape
# For example, for a high-resolution reconstruction:
volume_shape = optimal_shape

# For a lower resolution reconstruction that uses less memory:
# volume_shape = downsampled_shape  # Uncomment this if using the downsampled shape

# Determine which reconstruction method to use based on geometry
if 'geometry' in metadata and metadata['geometry'].get('GeometryType', '').lower() == 'cone':
    print("Using FDK reconstruction for cone beam geometry")
    # For FDK, we need a proper geometry dictionary
    reconstruction = fdk_reconstruction(projections, geometry, volume_shape)
else:
    print("Using Filtered Back Projection for parallel beam geometry")
    # For FBP, we can directly use the projections and angles
    reconstruction = filtered_backprojection(projections, angles, volume_shape)

print(f"Reconstruction complete. Output shape: {reconstruction.shape}")

## Visualize Reconstruction Results

Let's visualize some slices from our reconstruction.

In [ ]:
# Visualize central slices from the reconstructed volume
fig, axs = plt.subplots(1, 3, figsize=(15, 5))

# XY slice (top view)
z_middle = reconstruction.shape[2] // 2
axs[0].imshow(reconstruction[:, :, z_middle], cmap='gray')
axs[0].set_title(f"XY Slice (z={z_middle})")
axs[0].axis('off')

# XZ slice (side view)
y_middle = reconstruction.shape[1] // 2
axs[1].imshow(reconstruction[:, y_middle, :], cmap='gray')
axs[1].set_title(f"XZ Slice (y={y_middle})")
axs[1].axis('off')

# YZ slice (front view)
x_middle = reconstruction.shape[0] // 2
axs[2].imshow(reconstruction[x_middle, :, :], cmap='gray')
axs[2].set_title(f"YZ Slice (x={x_middle})")
axs[2].axis('off')

plt.tight_layout()
plt.show()

## Save Reconstruction Results

Finally, let's save our reconstruction results.

In [ ]:
from utils import save_volume_as_tiff_stack, save_numpy_as_vtk

# Create output directories
output_dir = "../results"
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# Save as TIFF stack
tiff_output_dir = os.path.join(output_dir, "tiff_slices")
print(f"Saving reconstruction as TIFF stack to {tiff_output_dir}...")
save_volume_as_tiff_stack(reconstruction, tiff_output_dir)

# Save as VTK file for 3D visualization
vtk_filename = os.path.join(output_dir, "reconstruction.vti")
print(f"Saving reconstruction as VTK file to {vtk_filename}...")

# Calculate spacing if we have physical detector size
spacing = (1.0, 1.0, 1.0)  # Default voxel spacing
if 'geometry' in metadata and 'detectorSize' in metadata['geometry'] and 'detectorPixel' in metadata['geometry']:
    detector_size = metadata['geometry']['detectorSize']  # Physical size in mm
    detector_pixels = metadata['geometry']['detectorPixel']  # Number of pixels
    
    # Calculate pixel size in mm
    if isinstance(detector_size, list) and isinstance(detector_pixels, list):
        pixel_size_x = detector_size[0] / detector_pixels[0]
        pixel_size_y = detector_size[1] / detector_pixels[1]
        spacing = (pixel_size_x, pixel_size_y, pixel_size_x)  # Assuming isotropic voxels
        print(f"Using voxel spacing: {spacing} mm")

save_numpy_as_vtk(reconstruction, vtk_filename, spacing=spacing)

print("Reconstruction results saved successfully!")

## Memory-Efficient 3D Visualization

For large volumes, direct visualization with PyVista can be memory-intensive. Here's a more memory-efficient approach.

In [ ]:
# Import PyVista for 3D visualization
import pyvista as pv

# Instead of loading the entire volume, load the VTK file directly
print("Loading the VTK file for visualization...")
reader = pv.get_reader(vtk_filename)
volume = reader.read()

# Downsample for visualization to reduce memory usage
sample_rate = (4, 4, 4)  # Sample every 4th point in each dimension
downsampled = volume.extract_subset(sample_rate=sample_rate)
print(f"Downsampled from {volume.dimensions} to {downsampled.dimensions} for visualization")

# Create a plotter
plotter = pv.Plotter()

# Add the volume with a suitable opacity transfer function
opacity = 'sigmoid_5'  # This often works well for CT data
plotter.add_volume(downsampled, cmap='viridis', opacity=opacity)

# Show the plot
plotter.show_grid()
plotter.show()